In [1]:
!pip install rapidfuzz sparse_dot_topn --quiet

import pandas as pd
import numpy as np
import re
import gc
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import awesome_cossim_topn
from rapidfuzz import fuzz, distance

# Настройка pandas для отображения
pd.set_option('display.max_colwidth', None)

print("Libraries loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.7/266.7 kB 11.4 MB/s eta 0:00:00
Libraries loaded.


In [2]:
orcs = pd.read_parquet('/kaggle/input/aim-2025-orcs/orcs.parquet')
employees = pd.read_parquet('/kaggle/input/aim-2025-orcs/employees.parquet')

In [3]:
# Настраиваем отображение 100 строк
pd.set_option('display.max_rows', 100)

cols_target = ['name', 'surname', 'fathername', 'inn', 'passport', 'birthdate']

print("=== АНАЛИЗ ПЕРЕД УНИФИКАЦИЕЙ ===")

def analyze_state(df, name):
    print(f"\n--- {name} ---")
    for col in cols_target:
        if col not in df.columns: continue
        
        # Текущий тип
        curr_type = df[col].dtype
        
        # Считаем реальные пропуски (null/nan/<NA>)
        null_count = df[col].isna().sum()
        
        print(f"[{col}] Тип: {curr_type} | Пустых значений: {null_count}")

analyze_state(orcs, "ORCS")
analyze_state(employees, "EMPLOYEES")

=== АНАЛИЗ ПЕРЕД УНИФИКАЦИЕЙ ===

--- ORCS ---
[name] Тип: string | Пустых значений: 0
[surname] Тип: string | Пустых значений: 0
[fathername] Тип: string | Пустых значений: 5
[inn] Тип: string | Пустых значений: 27156
[passport] Тип: string | Пустых значений: 6228
[birthdate] Тип: string | Пустых значений: 4300

--- EMPLOYEES ---
[name] Тип: object | Пустых значений: 13
[surname] Тип: object | Пустых значений: 5
[fathername] Тип: object | Пустых значений: 767
[inn] Тип: object | Пустых значений: 462402
[passport] Тип: object | Пустых значений: 342676
[birthdate] Тип: object | Пустых значений: 140722


In [4]:
def unify_types_inplace(df):
    for col in cols_target:
        if col in df.columns:
            # 1. Конвертируем в pandas string (это обрабатывает и object, и string)
            # 2. fillna("") заменяет <NA>, np.nan и None на пустую строку
            df[col] = df[col].astype("string").fillna("").str.lower()
    return df

print("Применяем унификацию типов...")
orcs = unify_types_inplace(orcs)
employees = unify_types_inplace(employees)

print("Готово. Повторная проверка:")
analyze_state(orcs, "ORCS")
analyze_state(employees, "EMPLOYEES")

Применяем унификацию типов...
Готово. Повторная проверка:

--- ORCS ---
[name] Тип: string | Пустых значений: 0
[surname] Тип: string | Пустых значений: 0
[fathername] Тип: string | Пустых значений: 0
[inn] Тип: string | Пустых значений: 0
[passport] Тип: string | Пустых значений: 0
[birthdate] Тип: string | Пустых значений: 0

--- EMPLOYEES ---
[name] Тип: string | Пустых значений: 0
[surname] Тип: string | Пустых значений: 0
[fathername] Тип: string | Пустых значений: 0
[inn] Тип: string | Пустых значений: 0
[passport] Тип: string | Пустых значений: 0
[birthdate] Тип: string | Пустых значений: 0


In [5]:
def preview_changes(df, col, clean_func):
    """
    1. Применяет функцию clean_func.
    2. Считает метрики уникальности (сжатие пространства имен).
    3. Показывает примеры изменений.
    """
    # Исходные данные
    original = df[col]
    
    # Применяем функцию
    modified = original.apply(clean_func)
    
    # Ищем разницу
    mask = original != modified
    count = mask.sum()
    
    # Считаем уникальные
    uniq_before = original.nunique()
    uniq_after = modified.nunique()
    diff_uniq = uniq_before - uniq_after
    
    print(f"=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ '{col}' ===")
    print(f"Строк затронуто: {count} (из {len(df)})")
    print(f"Уникальных значений ДО:    {uniq_before}")
    print(f"Уникальных значений ПОСЛЕ: {uniq_after}")
    
    if diff_uniq > 0:
        print(f"📉 Пространство имен сжалось на: {diff_uniq} значений (Дубликаты устранены)")
    elif diff_uniq < 0:
        print(f"📈 Пространство имен выросло на: {abs(diff_uniq)} (Странно! Проверь логику)")
    else:
        print(f"Пространство имен не изменилось.")
    
    if count > 0:
        print("\nПримеры изменений (первые 50):")
        diff_view = pd.DataFrame({
            'ДО': original[mask],
            'ПОСЛЕ': modified[mask]
        })
        display(diff_view.head(50))
    else:
        print("Изменений не найдено.")
    
    print("-" * 30)
    return mask, modified

In [6]:
def fix_yo(text):
    return text.replace('ё', 'е')

print("=== ПРЕДПРОСМОТР ЗАМЕНЫ Ё -> Е (ORCS) ===")
for col in ['name', 'surname', 'fathername']:
    if col in orcs.columns:
        # Используем функцию preview_changes из прошлого шага
        mask, modified = preview_changes(orcs, col, fix_yo)

print(">>> АНАЛИЗ ЗАМЕНЫ Ё -> Е (EMPLOYEES)")
for col in ['name', 'surname', 'fathername']:
    if col in employees.columns:
        preview_changes(employees, col, fix_yo)

=== ПРЕДПРОСМОТР ЗАМЕНЫ Ё -> Е (ORCS) ===
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'name' ===
Строк затронуто: 261 (из 47633)
Уникальных значений ДО:    32711
Уникальных значений ПОСЛЕ: 32689
📉 Пространство имен сжалось на: 22 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
52,ёалек`ей,еалек`ей
503,владиёиэ,владиеиэ
546,артём,артем
554,екётерина,екетерина
669,мёсис,месис
861,иъаё,иъае
1359,милёна,милена
1612,сёхге`,сехге`
1740,наёа,наеа
1778,дбанетё,дбанете


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'surname' ===
Строк затронуто: 458 (из 47633)
Уникальных значений ДО:    45432
Уникальных значений ПОСЛЕ: 45418
📉 Пространство имен сжалось на: 14 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
156,соложёнкова,соложенкова
339,ёиааев,еиааев
390,щёголева,щеголева
502,фёдорово,федорово
562,юживанширёв,юживанширев
687,жиенёва,жиенева
741,щербачёв,щербачев
840,ёонова,еонова
847,кута*ёв,кута*ев
928,валидёонбекйвк,валидеонбекйвк


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'fathername' ===
Строк затронуто: 314 (из 47633)
Уникальных значений ДО:    39727
Уникальных значений ПОСЛЕ: 39719
📉 Пространство имен сжалось на: 8 значений (Дубликаты устранены)

Примеры изменений (первые 50):


...

------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'surname' ===
Строк затронуто: 15417 (из 2011759)
Уникальных значений ДО:    921405
Уникальных значений ПОСЛЕ: 918526
📉 Пространство имен сжалось на: 2879 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
31,курёанова,куреанова
70,пестряёоъ,пестряеоъ
72,ортёмова,ортемова
184,сниёцкая,сниецкая
288,василёчилов,василечилов
538,глухарёв,глухарев
544,лещёво,лещево
618,тарёсовт,таресовт
785,яроваёроэая,яроваероэая
854,ёиоклагва,еиоклагва


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'fathername' ===
Строк затронуто: 11004 (из 2011759)
Уникальных значений ДО:    433969
Уникальных значений ПОСЛЕ: 432446
📉 Пространство имен сжалось на: 1523 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
170,"йлександровё,","йлександрове,"
288,арьэъовиё,арьэъовие
612,олегоёич,олегоеич
843,есмурзаевёч,есмурзаевеч
1280,каёедыбаеёин,каеедыбаееин
1334,эёнияович,эенияович
1364,ёятровна,еятровна
1419,фёдорона,федорона
1631,юёзыпович,юезыпович
2146,эрдогёнович,эрдогенович


------------------------------


In [7]:
print("=== ПРИМЕНЕНИЕ ФИКСА Ё -> Е ===")

def apply_yo_fix(df, df_name):
    changed_total = 0
    for col in ['name', 'surname', 'fathername']:
        if col not in df.columns: continue
        
        # Считаем, сколько меняется (для лога)
        original = df[col]
        df[col] = df[col].apply(fix_yo)
        
        diff = (original != df[col]).sum()
        changed_total += diff
        
    print(f"✅ {df_name}: Исправлено букв Ё в {changed_total} ячейках.")
    return df

# Применяем
orcs = apply_yo_fix(orcs, "Orcs")
employees = apply_yo_fix(employees, "Employees")

=== ПРИМЕНЕНИЕ ФИКСА Ё -> Е ===
✅ Orcs: Исправлено букв Ё в 1033 ячейках.
✅ Employees: Исправлено букв Ё в 34222 ячейках.


In [8]:
LATIN_TO_CYRILLIC = str.maketrans({
    'a': 'а',
    'b': 'в',
    'c': 'с',
    'e': 'е',
    'h': 'н',
    'i': 'и',
    'k': 'к',
    'm': 'м',
    'o': 'о',
    'p': 'р',
    't': 'т',
    'x': 'х',
    'y': 'у'
})
def fix_latin(text):
    return text.translate(LATIN_TO_CYRILLIC)

print("=== ЭТАП: ИСПРАВЛЕНИЕ ЛАТИНИЦЫ ===")

# 1. ПРЕВЬЮ
for df_name, df in [("Orcs", orcs), ("Employees", employees)]:
    print(f"\n--- {df_name} ---")
    for col in ['name', 'surname', 'fathername']:
        if col in df.columns:
            preview_changes(df, col, fix_latin)

=== ЭТАП: ИСПРАВЛЕНИЕ ЛАТИНИЦЫ ===

--- Orcs ---
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'name' ===
Строк затронуто: 17 (из 47633)
Уникальных значений ДО:    32689
Уникальных значений ПОСЛЕ: 32677
📉 Пространство имен сжалось на: 12 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
9827,валеpий,валерий
11538,галиhа,галина
12294,алекандpа,алекандра
12413,"cветлон,","светлон,"
13541,аhатолий,анатолий
15213,ромаh,роман
19578,александp,александр
24400,cергей,сергей
24650,атьяhа,атьяна
28786,алексаниp,алексанир


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'surname' ===
Строк затронуто: 3 (из 47633)
Уникальных значений ДО:    45418
Уникальных значений ПОСЛЕ: 45418
Пространство имен не изменилось.

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
18407,симаикh,симаикн
22466,оржаhова,оржанова
24114,куpицын,курицын


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'fathername' ===
Строк затронуто: 7 (из 47633)
Уникальных значений ДО:    39719
Уникальных значений ПОСЛЕ: 39714
📉 Пространство имен сжалось на: 5 значений (Дубликаты устранены)

Примеры изменений (первые 50):


...

------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'surname' ===
Строк затронуто: 14 (из 2011759)
Уникальных значений ДО:    918526
Уникальных значений ПОСЛЕ: 918518
📉 Пространство имен сжалось на: 8 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
140189,аржаhова,аржанова
179554,куpицын,курицын
224514,деpя.н,деря.н
421504,тюмиhо,тюмино
450651,семакиh,семакин
497686,димиртиеhко,димиртиенко
532976,деpябин,дерябин
842146,кузhецова,кузнецова
898971,симакиh,симакин
1535641,деpяби*а,деряби*а


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'fathername' ===
Строк затронуто: 50 (из 2011759)
Уникальных значений ДО:    432446
Уникальных значений ПОСЛЕ: 432411
📉 Пространство имен сжалось на: 35 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
20339,алеiоглы,алеиоглы
109528,васильевhа,васильевна
130916,ивоhович,ивонович
148984,ваеpьевич,ваерьевич
152092,викторовhа,викторовна
224070,bалеьревич,валеьревич
280460,hиолаевич,ниолаевич
337186,hикаhдровhа,никандровна
376792,петровhо,петровно
421504,алексаhдр`вhа,александр`вна


------------------------------


In [9]:
def apply_latin_fix(df, df_name):
    changed_total = 0
    for col in ['name', 'surname', 'fathername']:
        if col not in df.columns: continue
        
        original = df[col]
        df[col] = df[col].apply(fix_latin)
        
        diff = (original != df[col]).sum()
        changed_total += diff
        
    print(f"✅ {df_name}: Исправлена латиница в {changed_total} ячейках.")
    return df

# Применяем
orcs = apply_latin_fix(orcs, "Orcs")
employees = apply_latin_fix(employees, "Employees")

# ПРОВЕРКА: Осталась ли латиница?
print("\n=== КОНТРОЛЬНАЯ ПРОВЕРКА ЛАТИНИЦЫ ===")

for df_name, df in [("Orcs", orcs), ("Employees", employees)]:
    print(f"\n--- {df_name} ---")
    for col in ['name', 'surname', 'fathername']:
        if col not in df.columns: continue
        
        # Ищем латиницу
        mask = df[col].str.contains(r'[a-zA-Z]', regex=True)
        count = mask.sum()
        
        if count > 0:
            print(f"⚠️ {col}: Найдено {count} строк с остатками латиницы!")
            print(f"   Примеры: {df.loc[mask, col].unique()[:5]}")
        else:
            print(f"✅ {col}: Латиницы нет.")

✅ Orcs: Исправлена латиница в 27 ячейках.
✅ Employees: Исправлена латиница в 262 ячейках.

=== КОНТРОЛЬНАЯ ПРОВЕРКА ЛАТИНИЦЫ ===

--- Orcs ---
✅ name: Латиницы нет.
✅ surname: Латиницы нет.
✅ fathername: Латиницы нет.

--- Employees ---
✅ name: Латиницы нет.
✅ surname: Латиницы нет.
⚠️ fathername: Найдено 4 строк с остатками латиницы!
   Примеры: ['н\x7f. л  )&#      ^ :nеr< 9' 'н\x7f.л  е)&#      ^ :nез< 9'
 'н\x7f.л   )&#    а ^ :nеr< 9' ')&# ^ :nеr< 9 ишгв< 9 жан']


In [10]:
print("=== 1. ПРОВЕРКА ЛИШНИХ ПРОБЕЛОВ (TRIM) ===")

for df_name, df in [("Orcs", orcs), ("Employees", employees)]:
    for col in ['name', 'surname', 'fathername']:
        if col not in df.columns: continue
        
        # Сравниваем оригинал и strip-версию
        s = df[col]
        # strip() убирает пробелы в начале и конце
        mask_spaces = s != s.str.strip()
        count = mask_spaces.sum()
        
        if count > 0:
            print(f"⚠️ {df_name}.{col}: Найдено {count} строк с лишними пробелами по краям.")
            # Показываем, чтобы было видно (обернем в кавычки)
            examples = s[mask_spaces].head(3).tolist()
            print(f"   Примеры: {[f'|{x}|' for x in examples]}")
        else:
            print(f"✅ {df_name}.{col}: Лишних пробелов по краям нет.")



=== 1. ПРОВЕРКА ЛИШНИХ ПРОБЕЛОВ (TRIM) ===
✅ Orcs.name: Лишних пробелов по краям нет.
✅ Orcs.surname: Лишних пробелов по краям нет.
⚠️ Orcs.fathername: Найдено 16 строк с лишними пробелами по краям.
   Примеры: ['| фонасьевич|', '|фахед* |', '|виктор |']
✅ Employees.name: Лишних пробелов по краям нет.
✅ Employees.surname: Лишних пробелов по краям нет.
⚠️ Employees.fathername: Найдено 52 строк с лишними пробелами по краям.
   Примеры: ['|сами |', '| а анатольевеч|', '| фюря александровна|']


In [11]:
print("=== ПРИМЕНЕНИЕ TRIM (STRIP) ===")

cols_to_trim = ['name', 'surname', 'fathername']

for df_name, df in [("Orcs", orcs), ("Employees", employees)]:
    for col in cols_to_trim:
        if col not in df.columns: continue
        
        original = df[col]
        # Применяем strip
        df[col] = df[col].str.strip()
        
        # Статистика
        diff = (original != df[col]).sum()
        
    print(f"✅ {df_name}: Лишние пробелы удалены.")

=== ПРИМЕНЕНИЕ TRIM (STRIP) ===
✅ Orcs: Лишние пробелы удалены.
✅ Employees: Лишние пробелы удалены.


In [12]:
print("\n=== 2. ПРОВЕРКА ПЛЕЙСХОЛДЕРОВ (.,_?) ===")

# Паттерн: ищем точку, запятую, подчеркивание, вопрос, восклицательный знак
# Исключаем дефис, так как он легален в именах
PLACEHOLDER_REGEX = r'[.,_?!:;+]'

for df_name, df in [("Orcs", orcs), ("Employees", employees)]:
    for col in ['name', 'surname', 'fathername']:
        if col not in df.columns: continue
        
        s = df[col]
        mask_placeholders = s.str.contains(PLACEHOLDER_REGEX, regex=True)
        count = mask_placeholders.sum()
        
        if count > 0:
            print(f"⚠️ {df_name}.{col}: Найдено {count} строк с символами-заменителями.")
            print(f"   Примеры: {s[mask_placeholders].unique()[:10]}")
            
            # Посмотрим, какие именно символы там встречаются
            found_chars = set(re.findall(PLACEHOLDER_REGEX, "".join(s[mask_placeholders].head(100))))
            print(f"   Найденные символы: {found_chars}")
        else:
            print(f"✅ {df_name}.{col}: Символов-заменителей не найдено.")


=== 2. ПРОВЕРКА ПЛЕЙСХОЛДЕРОВ (.,_?) ===
⚠️ Orcs.name: Найдено 2041 строк с символами-заменителями.
   Примеры: ['эмир,ан' 'еле,а' 'элму.' 'ве,а' 'вл.вдимир' 'ан,а' 'т.джимурод'
 'м.хавер' 'га.ибальди' 'вее,мин']
   Найденные символы: {',', '.'}
⚠️ Orcs.surname: Найдено 3486 строк с символами-заменителями.
   Примеры: ['чуга,ва' 'чорша.биев' 'изо.ченко' 'димит,иу*' 'п.хитна' 'вак.люк'
 'а,танев' 'ереми.а' 'е.зов' 'т,зиктинова']
   Найденные символы: {',', '.'}
⚠️ Orcs.fathername: Найдено 3586 строк с символами-заменителями.
   Примеры: ['фаинуров.ч' 'нико.аевич' 'сарг,сович' 'в.кторвона' 'хъссейн .афаат'
 'саяд,ин-оглы' 'фдео.овна' 'кап.талинович' 'мих.йлович'
 'рауль анхе.ь **********']
   Найденные символы: {',', '.'}
⚠️ Employees.name: Найдено 64803 строк с символами-заменителями.
   Примеры: ['ан,рей' 'анато,ий' 'евге.ий' 'а.дрй' 'ма,ия' 'лиди,' 'о.ег' 'людми.а'
 'ел,на' 'любо.ь']
   Найденные символы: {',', '.'}
⚠️ Employees.surname: Найдено 122676 строк с символами-заменителями.

In [13]:
print("=== СОЗДАНИЕ ТАБЛИЦЫ ОЧИСТКИ СИМВОЛОВ ===")

# 1. Настраиваем группы
TO_STAR = ".,?'`_"           # Превращаем в *
TO_DELETE = "0123456789nr!\"#$%&()+/:;<=>@[\]^{|}~"

print(f"В звездочку (*): [{TO_STAR}]")
print(f"В удаление:      [{TO_DELETE}]")

# 2. Компилируем таблицу перевода
TRANS_TABLE = {}

for char in TO_STAR:
    TRANS_TABLE[ord(char)] = '*'
    
for char in TO_DELETE:
    TRANS_TABLE[ord(char)] = None

def clean_symbols(text):
    return text.translate(TRANS_TABLE)

# 3. ПРЕВЬЮ
print("\n=== ПРЕДПРОСМОТР ОЧИСТКИ СИМВОЛОВ ===")
for df_name, df in [("Orcs", orcs), ("Employees", employees)]:
    print(f"\n--- {df_name} ---")
    for col in ['name', 'surname', 'fathername']:
        if col in df.columns:
            preview_changes(df, col, clean_symbols)

=== СОЗДАНИЕ ТАБЛИЦЫ ОЧИСТКИ СИМВОЛОВ ===
В звездочку (*): [.,?'`_]
В удаление:      [0123456789nr!"#$%&()+/:;<=>@[\]^{|}~]

=== ПРЕДПРОСМОТР ОЧИСТКИ СИМВОЛОВ ===

--- Orcs ---
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'name' ===
Строк затронуто: 2998 (из 47633)
Уникальных значений ДО:    32677
Уникальных значений ПОСЛЕ: 32398
📉 Пространство имен сжалось на: 279 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
0,габ`узхун,габ*узхун
17,"эмир,ан",эмир*ан
18,"еле,а",еле*а
47,элму.,элму*
50,русл`н,русл*н
52,еалек`ей,еалек*ей
54,"ве,а",ве*а
61,св`тлана,св*тлана
64,вл.вдимир,вл*вдимир
68,еле`а,еле*а


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'surname' ===
Строк затронуто: 5123 (из 47633)
Уникальных значений ДО:    45418
Уникальных значений ПОСЛЕ: 45391
📉 Пространство имен сжалось на: 27 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
12,про`ор,про*ор
25,"чуга,ва",чуга*ва
38,чорша.биев,чорша*биев
43,изо.ченко,изо*ченко
57,"димит,иу*",димит*иу*
81,п.хитна,п*хитна
91,вак.люк,вак*люк
98,"а,танев",а*танев
104,ереми.а,ереми*а
114,е.зов,е*зов


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'fathername' ===
Строк затронуто: 5339 (из 47633)
Уникальных значений ДО:    39712
Уникальных значений ПОСЛЕ: 39325
📉 Пространство имен сжалось на: 387 значений (Дубликаты устранены)

Примеры изменений (первые 50):


...

------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'surname' ===
Строк затронуто: 183014 (из 2011759)
Уникальных значений ДО:    918518
Уникальных значений ПОСЛЕ: 885674
📉 Пространство имен сжалось на: 32844 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
15,"жапарку,ов",жапарку*ов
21,власо`а,власо*а
52,патрекеев.,патрекеев*
84,ма`аев,ма*аев
86,хохлов.,хохлов*
103,п`алов,п*алов
134,слу`ин,слу*ин
140,"макс,мов",макс*мов
142,"ни,шкова",ни*шкова
144,"к,стаб",к*стаб


------------------------------
=== ПРЕДПРОСМОТР ИЗМЕНЕНИЙ ДЛЯ 'fathername' ===
Строк затронуто: 184123 (из 2011759)
Уникальных значений ДО:    432393
Уникальных значений ПОСЛЕ: 412877
📉 Пространство имен сжалось на: 19516 значений (Дубликаты устранены)

Примеры изменений (первые 50):


,ДО,ПОСЛЕ
1,"али,вич",али*вич
2,вяч.славовеч,вяч*славовеч
18,"леонидо,на",леонидо*на
21,васи.еьвна,васи*еьвна
41,евге`ьевна,евге*ьевна
47,"олимжанови,",олимжанови*
99,сергее`еч,сергее*еч
110,нагемуллови.,нагемуллови*
115,"михайлов,ч",михайлов*ч
162,абус`ттаровна,абус*ттаровна


------------------------------


In [14]:
print("=== ПРИМЕНЕНИЕ ОЧИСТКИ СИМВОЛОВ ===")

def apply_symbol_clean(df, df_name):
    changed_total = 0
    for col in ['name', 'surname', 'fathername']:
        if col not in df.columns: continue
        
        original = df[col]
        df[col] = df[col].apply(clean_symbols)
        
        diff = (original != df[col]).sum()
        changed_total += diff
        
    print(f"✅ {df_name}: Символы очищены в {changed_total} ячейках.")
    return df

orcs = apply_symbol_clean(orcs, "Orcs")
employees = apply_symbol_clean(employees, "Employees")

# Контроль на "выжившие" плохие символы
print("=== КОНТРОЛЬ ЧИСТОТЫ (ПОИСК НЕКИРИЛЛИЧЕСКИХ СИМВОЛОВ) ===")

def get_alien_chars(df_list, cols):
    # 1. Формируем белый список
    # а-я (коды 1072-1103)
    allowed = set([chr(i) for i in range(ord('а'), ord('я') + 1)])
    # Добавляем легальные спецсимволы
    allowed.update([' ', '-', '*'])
    
    # 2. Собираем все символы из датафреймов
    found_chars = set()
    
    for df in df_list:
        for col in cols:
            if col not in df.columns: continue
            
            # Берем уникальные значения, склеиваем в одну строку
            # (dropna + unique для скорости)
            text_blob = "".join(df[col].dropna().unique())
            found_chars.update(text_blob)
            
    # 3. Вычитаем разрешенные из найденных
    aliens = found_chars - allowed
    
    return sorted(list(aliens))

# Запускаем проверку
alien_symbols = get_alien_chars([orcs, employees], ['name', 'surname', 'fathername'])

print(f"Найдено посторонних символов: {len(alien_symbols)}")

if len(alien_symbols) > 0:
    print("Список чужаков:")
    for char in alien_symbols:
        # Выводим сам символ и его код (чтобы понять, что это за пробел или знак)
        print(f"--> '{char}' (Code: {ord(char)})")
else:
    print("✅ Идеальная чистота! Только кириллица, пробелы, дефисы и звездочки.")

=== ПРИМЕНЕНИЕ ОЧИСТКИ СИМВОЛОВ ===
✅ Orcs: Символы очищены в 13460 ячейках.
✅ Employees: Символы очищены в 463919 ячейках.
=== КОНТРОЛЬ ЧИСТОТЫ (ПОИСК НЕКИРИЛЛИЧЕСКИХ СИМВОЛОВ) ===
Найдено посторонних символов: 0
✅ Идеальная чистота! Только кириллица, пробелы, дефисы и звездочки.


In [15]:
import os

print("=== СОХРАНЕНИЕ РЕЗУЛЬТАТОВ (CHECKPOINT) ===")


orcs.to_parquet('orcs_clean.parquet', index=True) 
employees.to_parquet('employees_clean.parquet', index=True)

print(f"Файлы сохранены:")
print(f"1. orcs_clean.parquet ({len(orcs)} строк)")
print(f"2. employees_clean.parquet ({len(employees)} строк)")

=== СОХРАНЕНИЕ РЕЗУЛЬТАТОВ (CHECKPOINT) ===
Файлы сохранены:
1. orcs_clean.parquet (47633 строк)
2. employees_clean.parquet (2011759 строк)
